In [1]:
import numpy as np
import pandas as pd
import sklearn
import transformers
import torchvision

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [2]:
from __future__ import absolute_import, division, print_function
import logging
import numpy as np
import random
import sys
import time
import torch

from argparse import Namespace
from torch.utils.data import (DataLoader, RandomSampler, SequentialSampler,
                              TensorDataset)
from tqdm import tqdm
from transformers import (BertConfig, BertForSequenceClassification, BertTokenizer,)
from transformers import glue_compute_metrics as compute_metrics
from transformers import glue_output_modes as output_modes
from transformers import glue_processors as processors
from transformers import glue_convert_examples_to_features as convert_examples_to_features

logger = logging.getLogger(__name__)
logging.basicConfig(format = '%(asctime)s - %(levelname)s - %(name)s -   %(message)s',
                    datefmt = '%m/%d/%Y %H:%M:%S',
                    level = logging.WARN)

logging.getLogger("transformers.modeling_utils").setLevel(
   logging.WARN)  # Reduce logging

print(torch.__version__)

2026-01-28 08:22:55.506172: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-28 08:22:55.506322: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-28 08:22:55.634498: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


2.1.2+cpu


We set the number of threads to compare the single thread performance between FP32 and INT8 performance. In the end of the tutorial, the user can set other number of threads by building PyTorch with right parallel backend.

In [3]:
torch.set_num_threads(1)
print(torch.__config__.parallel_info())

ATen/Parallel:
	at::get_num_threads() : 1
	at::get_num_interop_threads() : 2
OpenMP 201511 (a.k.a. OpenMP 4.5)
	omp_get_max_threads() : 1
Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
	mkl_get_max_threads() : 1
Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
std::thread::hardware_concurrency() : 4
Environment variables:
	OMP_NUM_THREADS : [not set]
	MKL_NUM_THREADS : [not set]
ATen parallel backend: OpenMP



# Download Dataset

In [4]:
!pwd
!ls
!wget https://gist.githubusercontent.com/W4ngatang/60c2bdb54d156a41194446737ce03e2e/raw/17b8dd0d724281ed7c3b2aeeda662b92809aadd5/download_glue_data.py
!python download_glue_data.py --data_dir='glue_data' --tasks='MRPC'
!ls glue_data/MRPC

/kaggle/working
__notebook__.ipynb
--2026-01-28 08:23:08--  https://gist.githubusercontent.com/W4ngatang/60c2bdb54d156a41194446737ce03e2e/raw/17b8dd0d724281ed7c3b2aeeda662b92809aadd5/download_glue_data.py
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8225 (8.0K) [text/plain]
Saving to: 'download_glue_data.py'

download_glue_data. 100%[===================>]   8.03K  --.-KB/s    in 0.001s  

2026-01-28 08:23:09 (11.3 MB/s) - 'download_glue_data.py' saved [8225/8225]

Processing MRPC...
Local MRPC data not specified, downloading data from https://dl.fbaipublicfiles.com/senteval/senteval_data/msr_paraphrase_train.txt
	Completed!
dev.tsv      msr_paraphrase_test.txt   test.tsv
dev_ids.tsv  msr_paraphrase_train.txt  train.tsv


# Import Fine-tune the BERT model

In [5]:
!wget https://download.pytorch.org/tutorial/MRPC.zip
!unzip MRPC.zip
!ls
!pwd

--2026-01-28 08:23:13--  https://download.pytorch.org/tutorial/MRPC.zip
Resolving download.pytorch.org (download.pytorch.org)... 3.169.137.77, 3.169.137.102, 3.169.137.100, ...
Connecting to download.pytorch.org (download.pytorch.org)|3.169.137.77|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 405365618 (387M) [application/zip]
Saving to: 'MRPC.zip'

MRPC.zip            100%[===================>] 386.59M  25.7MB/s    in 16s     

2026-01-28 08:23:30 (23.6 MB/s) - 'MRPC.zip' saved [405365618/405365618]

Archive:  MRPC.zip
   creating: MRPC/
 extracting: MRPC/added_tokens.json  
  inflating: MRPC/tokenizer_config.json  
  inflating: MRPC/special_tokens_map.json  
  inflating: MRPC/config.json        
  inflating: MRPC/training_args.bin  
  inflating: MRPC/vocab.txt          
  inflating: MRPC/pytorch_model.bin  
MRPC  MRPC.zip	__notebook__.ipynb  download_glue_data.py  glue_data
/kaggle/working


# Set global configurations

In [6]:
configs = Namespace()

# The output directory for the fine-tuned model.
configs.output_dir = "/kaggle/working/MRPC"
# configs.output_dir = "./MRPC/"

# The data directory for the MRPC task in the GLUE benchmark.
configs.data_dir = "/kaggle/working/glue_data/MRPC"

# The model name or path for the pre-trained model.
configs.model_name_or_path = "bert-base-uncased"
# The maximum length of an input sequence
configs.max_seq_length = 128

# Prepare GLUE task.
configs.task_name = "MRPC".lower()
configs.processor = processors[configs.task_name]()
configs.output_mode = output_modes[configs.task_name]
configs.label_list = configs.processor.get_labels()
configs.model_type = "bert".lower()
configs.do_lower_case = True

# Set the device, batch size, topology, and caching flags.
configs.device = "cpu"
configs.per_gpu_eval_batch_size = 8
configs.n_gpu = 0
configs.local_rank = -1
configs.overwrite_cache = False


# Set random seed for reproducibility.
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
set_seed(42)

/opt/conda/lib/python3.10/site-packages/transformers/data/processors/glue.py:174: FutureWarning: This processor will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING.format("processor"), FutureWarning)


# Load the fine-tuned BERT model

In [7]:
print(configs.output_dir) 
tokenizer = BertTokenizer.from_pretrained(
    configs.output_dir, do_lower_case=configs.do_lower_case)

model = BertForSequenceClassification.from_pretrained(configs.output_dir)
model.to(configs.device)

/opt/conda/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


/kaggle/working/MRPC


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [8]:
from prettytable import PrettyTable

def count_parameters(model):
    table = PrettyTable(["Modules", "Parameters"])
    total_params = 0
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        params = parameter.numel()
        table.add_row([name, params])
        total_params += params
    print(table)
    print(f"Total Trainable Params: {total_params}")
    return total_params
    
count_parameters(model)

+---------------------------------------------------------+------------+
|                         Modules                         | Parameters |
+---------------------------------------------------------+------------+
|          bert.embeddings.word_embeddings.weight         |  23440896  |
|        bert.embeddings.position_embeddings.weight       |   393216   |
|       bert.embeddings.token_type_embeddings.weight      |    1536    |
|             bert.embeddings.LayerNorm.weight            |    768     |
|              bert.embeddings.LayerNorm.bias             |    768     |
|     bert.encoder.layer.0.attention.self.query.weight    |   589824   |
|      bert.encoder.layer.0.attention.self.query.bias     |    768     |
|      bert.encoder.layer.0.attention.self.key.weight     |   589824   |
|       bert.encoder.layer.0.attention.self.key.bias      |    768     |
|     bert.encoder.layer.0.attention.self.value.weight    |   589824   |
|      bert.encoder.layer.0.attention.self.value.bi

109483778

In [9]:
def evaluate(args, model, tokenizer, prefix=""):
    # Loop to handle MNLI double evaluation (matched, mis-matched)
    eval_task_names = ("mnli", "mnli-mm") if args.task_name == "mnli" else (args.task_name,)
    eval_outputs_dirs = (args.output_dir, args.output_dir + '-MM') if args.task_name == "mnli" else (args.output_dir,)

    results = {}
    for eval_task, eval_output_dir in zip(eval_task_names, eval_outputs_dirs):
        eval_dataset = load_and_cache_examples(args, eval_task, tokenizer, evaluate=True)

        if not os.path.exists(eval_output_dir) and args.local_rank in [-1, 0]:
            os.makedirs(eval_output_dir)

        args.eval_batch_size = args.per_gpu_eval_batch_size * max(1, args.n_gpu)
        # Note that DistributedSampler samples randomly
        eval_sampler = SequentialSampler(eval_dataset) if args.local_rank == -1 else DistributedSampler(eval_dataset)
        eval_dataloader = DataLoader(eval_dataset, sampler=eval_sampler, batch_size=args.eval_batch_size)

        # multi-gpu eval
        if args.n_gpu > 1:
            model = torch.nn.DataParallel(model)

        # Eval!
        logger.info("***** Running evaluation {} *****".format(prefix))
        logger.info("  Num examples = %d", len(eval_dataset))
        logger.info("  Batch size = %d", args.eval_batch_size)
        eval_loss = 0.0
        nb_eval_steps = 0
        preds = None
        out_label_ids = None
        for batch in tqdm(eval_dataloader, desc="Evaluating"):
            model.eval()
            batch = tuple(t.to(args.device) for t in batch)

            with torch.no_grad():
                inputs = {'input_ids':      batch[0],
                          'attention_mask': batch[1],
                          'labels':         batch[3]}
                if args.model_type != 'distilbert':
                    inputs['token_type_ids'] = batch[2] if args.model_type in ['bert', 'xlnet'] else None  # XLM, DistilBERT and RoBERTa don't use segment_ids
                outputs = model(**inputs)
                tmp_eval_loss, logits = outputs[:2]

                eval_loss += tmp_eval_loss.mean().item()
            nb_eval_steps += 1
            if preds is None:
                preds = logits.detach().cpu().numpy()
                out_label_ids = inputs['labels'].detach().cpu().numpy()
            else:
                preds = np.append(preds, logits.detach().cpu().numpy(), axis=0)
                out_label_ids = np.append(out_label_ids, inputs['labels'].detach().cpu().numpy(), axis=0)

        eval_loss = eval_loss / nb_eval_steps
        if args.output_mode == "classification":
            preds = np.argmax(preds, axis=1)
        elif args.output_mode == "regression":
            preds = np.squeeze(preds)
        result = compute_metrics(eval_task, preds, out_label_ids)
        results.update(result)

        output_eval_file = os.path.join(eval_output_dir, prefix, "eval_results.txt")
        with open(output_eval_file, "w") as writer:
            logger.info("***** Eval results {} *****".format(prefix))
            for key in sorted(result.keys()):
                logger.info("  %s = %s", key, str(result[key]))
                writer.write("%s = %s\n" % (key, str(result[key])))

    return results


def load_and_cache_examples(args, task, tokenizer, evaluate=False):
    if args.local_rank not in [-1, 0] and not evaluate:
        torch.distributed.barrier()  # Make sure only the first process in distributed training process the dataset, and the others will use the cache

    processor = processors[task]()
    output_mode = output_modes[task]
    # Load data features from cache or dataset file
    cached_features_file = os.path.join(args.data_dir, 'cached_{}_{}_{}_{}'.format(
        'dev' if evaluate else 'train',
        list(filter(None, args.model_name_or_path.split('/'))).pop(),
        str(args.max_seq_length),
        str(task)))
    if os.path.exists(cached_features_file) and not args.overwrite_cache:
        logger.info("Loading features from cached file %s", cached_features_file)
        features = torch.load(cached_features_file)
    else:
        logger.info("Creating features from dataset file at %s", args.data_dir)
        label_list = processor.get_labels()
        if task in ['mnli', 'mnli-mm'] and args.model_type in ['roberta']:
            # HACK(label indices are swapped in RoBERTa pretrained model)
            label_list[1], label_list[2] = label_list[2], label_list[1] 
        examples = processor.get_dev_examples(args.data_dir) if evaluate else processor.get_train_examples(args.data_dir)
        features = convert_examples_to_features(examples,
                                                tokenizer,
                                                label_list=label_list,
                                                max_length=args.max_seq_length,
                                                output_mode=output_mode,
#                                                 pad_on_left=bool(args.model_type in ['xlnet']),                 # pad on the left for xlnet
#                                                 pad_token=tokenizer.convert_tokens_to_ids([tokenizer.pad_token])[0],
#                                                 pad_token_segment_id=4 if args.model_type in ['xlnet'] else 0,
        )
        if args.local_rank in [-1, 0]:
            logger.info("Saving features into cached file %s", cached_features_file)
            torch.save(features, cached_features_file)

    if args.local_rank == 0 and not evaluate:
        torch.distributed.barrier()  # Make sure only the first process in distributed training process the dataset, and the others will use the cache

    # Convert to Tensors and build dataset
    all_input_ids = torch.tensor([f.input_ids for f in features], dtype=torch.long)
    all_attention_mask = torch.tensor([f.attention_mask for f in features], dtype=torch.long)
    all_token_type_ids = torch.tensor([f.token_type_ids for f in features], dtype=torch.long)
    if output_mode == "classification":
        all_labels = torch.tensor([f.label for f in features], dtype=torch.long)
    elif output_mode == "regression":
        all_labels = torch.tensor([f.label for f in features], dtype=torch.float)

    dataset = TensorDataset(all_input_ids, all_attention_mask, all_token_type_ids, all_labels)
    return dataset


In [10]:
from prettytable import PrettyTable

def count_parameters(model):
    table = PrettyTable(["Modules", "Parameters"])
    total_params = 0
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        params = parameter.numel()
        table.add_row([name, params])
        total_params += params
    print(table)
    print(f"Total Trainable Params: {total_params}")
    return total_params
    
count_parameters(model)

+---------------------------------------------------------+------------+
|                         Modules                         | Parameters |
+---------------------------------------------------------+------------+
|          bert.embeddings.word_embeddings.weight         |  23440896  |
|        bert.embeddings.position_embeddings.weight       |   393216   |
|       bert.embeddings.token_type_embeddings.weight      |    1536    |
|             bert.embeddings.LayerNorm.weight            |    768     |
|              bert.embeddings.LayerNorm.bias             |    768     |
|     bert.encoder.layer.0.attention.self.query.weight    |   589824   |
|      bert.encoder.layer.0.attention.self.query.bias     |    768     |
|      bert.encoder.layer.0.attention.self.key.weight     |   589824   |
|       bert.encoder.layer.0.attention.self.key.bias      |    768     |
|     bert.encoder.layer.0.attention.self.value.weight    |   589824   |
|      bert.encoder.layer.0.attention.self.value.bi

109483778

# Define the tokenize and evaluation function

# 1. Apply the dynamic quantization
We call torch.quantization.quantize_dynamic on the model to apply the dynamic quantization on the HuggingFace BERT model. Specifically,

We specify that we want the torch.nn.Linear modules in our model to be quantized;
We specify that we want weights to be converted to quantized int8 values.

In [11]:
# Define quantization configuration
model.train()
# model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')
model.qconfig = torch.quantization.float_qparams_weight_only_qconfig

# Prepare the model for QAT
model = torch.quantization.prepare_qat(model, inplace=False)

quantized_model = torch.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)
print(quantized_model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): QuantizedEmbedding(num_embeddings=30522, embedding_dim=768, dtype=torch.quint8, qscheme=torch.per_channel_affine_float_qparams)
      (position_embeddings): QuantizedEmbedding(num_embeddings=512, embedding_dim=768, dtype=torch.quint8, qscheme=torch.per_channel_affine_float_qparams)
      (token_type_embeddings): QuantizedEmbedding(num_embeddings=2, embedding_dim=768, dtype=torch.quint8, qscheme=torch.per_channel_affine_float_qparams)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(
                in_features=768, out_features=768, bias=True
                (weight_fake_quant): PerChannelMinMaxObserver(min

In [12]:
from prettytable import PrettyTable

def count_parameters(model):
    table = PrettyTable(["Modules", "Parameters"])
    total_params = 0
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        params = parameter.numel()
        table.add_row([name, params])
        total_params += params
    print(table)
    print(f"Total Trainable Params: {total_params}")
    return total_params
    
count_parameters(quantized_model)

+---------------------------------------------------------+------------+
|                         Modules                         | Parameters |
+---------------------------------------------------------+------------+
|             bert.embeddings.LayerNorm.weight            |    768     |
|              bert.embeddings.LayerNorm.bias             |    768     |
|     bert.encoder.layer.0.attention.self.query.weight    |   589824   |
|      bert.encoder.layer.0.attention.self.query.bias     |    768     |
|      bert.encoder.layer.0.attention.self.key.weight     |   589824   |
|       bert.encoder.layer.0.attention.self.key.bias      |    768     |
|     bert.encoder.layer.0.attention.self.value.weight    |   589824   |
|      bert.encoder.layer.0.attention.self.value.bias     |    768     |
|    bert.encoder.layer.0.attention.output.dense.weight   |   589824   |
|     bert.encoder.layer.0.attention.output.dense.bias    |    768     |
|  bert.encoder.layer.0.attention.output.LayerNorm.

85648130

In [13]:

quantized_model_2 = torch.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.float16
)
print(quantized_model_2)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): QuantizedEmbedding(num_embeddings=30522, embedding_dim=768, dtype=torch.quint8, qscheme=torch.per_channel_affine_float_qparams)
      (position_embeddings): QuantizedEmbedding(num_embeddings=512, embedding_dim=768, dtype=torch.quint8, qscheme=torch.per_channel_affine_float_qparams)
      (token_type_embeddings): QuantizedEmbedding(num_embeddings=2, embedding_dim=768, dtype=torch.quint8, qscheme=torch.per_channel_affine_float_qparams)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(
                in_features=768, out_features=768, bias=True
                (weight_fake_quant): PerChannelMinMaxObserver(min

In [14]:
from prettytable import PrettyTable

def count_parameters(model):
    table = PrettyTable(["Modules", "Parameters"])
    total_params = 0
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        params = parameter.numel()
        table.add_row([name, params])
        total_params += params
    print(table)
    print(f"Total Trainable Params: {total_params}")
    return total_params
    
count_parameters(quantized_model_2)

+---------------------------------------------------------+------------+
|                         Modules                         | Parameters |
+---------------------------------------------------------+------------+
|             bert.embeddings.LayerNorm.weight            |    768     |
|              bert.embeddings.LayerNorm.bias             |    768     |
|     bert.encoder.layer.0.attention.self.query.weight    |   589824   |
|      bert.encoder.layer.0.attention.self.query.bias     |    768     |
|      bert.encoder.layer.0.attention.self.key.weight     |   589824   |
|       bert.encoder.layer.0.attention.self.key.bias      |    768     |
|     bert.encoder.layer.0.attention.self.value.weight    |   589824   |
|      bert.encoder.layer.0.attention.self.value.bias     |    768     |
|    bert.encoder.layer.0.attention.output.dense.weight   |   589824   |
|     bert.encoder.layer.0.attention.output.dense.bias    |    768     |
|  bert.encoder.layer.0.attention.output.LayerNorm.

85648130

# 2. Apply the dynamic quantization with mixed precision

In [15]:
# from torch.cuda.amp import autocast

quantized_model_2 = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},  # Specify layers to quantize
    dtype=torch.float16
)

# # Perform inference with mixed precision
# def infer_with_mixed_precision(model, data_loader):
#     model.eval()
#     results = []
#     with torch.no_grad():
#         for inputs in data_loader:
#             with autocast():
#                 outputs = model(inputs)
#                 results.append(outputs)
#     return results

# Load the data
# data_loader = torch.utils.data.DataLoader(eval_dataset, batch_size=32)

# Perform inference
# results = infer_with_mixed_precision(quantized_model_2, data_loader)

# Print results
# print(results)

# quantized_model_2 = torch.quantization.quantize_dynamic(
#     model, {torch.nn.Linear}, dtype=torch.qint8
# )
print(quantized_model_2)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): QuantizedEmbedding(num_embeddings=30522, embedding_dim=768, dtype=torch.quint8, qscheme=torch.per_channel_affine_float_qparams)
      (position_embeddings): QuantizedEmbedding(num_embeddings=512, embedding_dim=768, dtype=torch.quint8, qscheme=torch.per_channel_affine_float_qparams)
      (token_type_embeddings): QuantizedEmbedding(num_embeddings=2, embedding_dim=768, dtype=torch.quint8, qscheme=torch.per_channel_affine_float_qparams)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(
                in_features=768, out_features=768, bias=True
                (weight_fake_quant): PerChannelMinMaxObserver(min

# Check the model size

In [16]:
from prettytable import PrettyTable

def count_parameters(model):
    table = PrettyTable(["Modules", "Parameters"])
    total_params = 0
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        params = parameter.numel()
        table.add_row([name, params])
        total_params += params
    print(table)
    print(f"Total Trainable Params: {total_params}")
    return total_params
    
count_parameters(quantized_model_2)

+---------------------------------------------------------+------------+
|                         Modules                         | Parameters |
+---------------------------------------------------------+------------+
|             bert.embeddings.LayerNorm.weight            |    768     |
|              bert.embeddings.LayerNorm.bias             |    768     |
|     bert.encoder.layer.0.attention.self.query.weight    |   589824   |
|      bert.encoder.layer.0.attention.self.query.bias     |    768     |
|      bert.encoder.layer.0.attention.self.key.weight     |   589824   |
|       bert.encoder.layer.0.attention.self.key.bias      |    768     |
|     bert.encoder.layer.0.attention.self.value.weight    |   589824   |
|      bert.encoder.layer.0.attention.self.value.bias     |    768     |
|    bert.encoder.layer.0.attention.output.dense.weight   |   589824   |
|     bert.encoder.layer.0.attention.output.dense.bias    |    768     |
|  bert.encoder.layer.0.attention.output.LayerNorm.

85648130

In [17]:
def print_size_of_model(model):
    torch.save(model.state_dict(), "temp.p")
    print('Size (MB):', os.path.getsize("temp.p")/1e6)
    os.remove('temp.p')

print_size_of_model(model)
print_size_of_model(quantized_model)
print_size_of_model(quantized_model_2)

Size (MB): 438.081423
Size (MB): 366.814149
Size (MB): 366.814149


# Evaluate the inference accuracy and time

In [18]:
def time_model_evaluation(model, configs, tokenizer):
    eval_start_time = time.time()
    result = evaluate(configs, model, tokenizer, prefix="")
    eval_end_time = time.time()
    eval_duration_time = eval_end_time - eval_start_time
    print(result)
    print("Evaluate total time (seconds): {0:.1f}".format(eval_duration_time))

In [19]:
# Evaluate the original FP32 BERT model
time_model_evaluation(model, configs, tokenizer)

/opt/conda/lib/python3.10/site-packages/transformers/data/processors/glue.py:66: FutureWarning: This function will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING.format("function"), FutureWarning)
Evaluating: 100%|██████████| 51/51 [02:40<00:00,  3.15s/it]

{'acc': 0.8602941176470589, 'f1': 0.9018932874354562, 'acc_and_f1': 0.8810937025412575}
Evaluate total time (seconds): 161.3



/opt/conda/lib/python3.10/site-packages/transformers/data/metrics/__init__.py:61: FutureWarning: This metric will be removed from the library soon, metrics should be handled with the 🤗 Evaluate library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING, FutureWarning)
/opt/conda/lib/python3.10/site-packages/transformers/data/metrics/__init__.py:37: FutureWarning: This metric will be removed from the library soon, metrics should be handled with the 🤗 Evaluate library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING, FutureWarning)
/opt/conda/lib/python3.10/site-packages/transformers/data/metrics/__init__.py:31: FutureWarning: This metric will be removed from the library soon, metrics should be han

In [20]:
time_model_evaluation(quantized_model, configs, tokenizer)

/opt/conda/lib/python3.10/site-packages/transformers/data/processors/glue.py:174: FutureWarning: This processor will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING.format("processor"), FutureWarning)
Evaluating: 100%|██████████| 51/51 [02:40<00:00,  3.15s/it]

{'acc': 0.8602941176470589, 'f1': 0.9018932874354562, 'acc_and_f1': 0.8810937025412575}
Evaluate total time (seconds): 160.7



/opt/conda/lib/python3.10/site-packages/transformers/data/metrics/__init__.py:61: FutureWarning: This metric will be removed from the library soon, metrics should be handled with the 🤗 Evaluate library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING, FutureWarning)
/opt/conda/lib/python3.10/site-packages/transformers/data/metrics/__init__.py:37: FutureWarning: This metric will be removed from the library soon, metrics should be handled with the 🤗 Evaluate library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING, FutureWarning)
/opt/conda/lib/python3.10/site-packages/transformers/data/metrics/__init__.py:31: FutureWarning: This metric will be removed from the library soon, metrics should be han

In [21]:
time_model_evaluation(quantized_model_2, configs, tokenizer)

/opt/conda/lib/python3.10/site-packages/transformers/data/processors/glue.py:174: FutureWarning: This processor will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING.format("processor"), FutureWarning)
Evaluating: 100%|██████████| 51/51 [02:40<00:00,  3.15s/it]

{'acc': 0.8602941176470589, 'f1': 0.9018932874354562, 'acc_and_f1': 0.8810937025412575}
Evaluate total time (seconds): 160.7



/opt/conda/lib/python3.10/site-packages/transformers/data/metrics/__init__.py:61: FutureWarning: This metric will be removed from the library soon, metrics should be handled with the 🤗 Evaluate library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING, FutureWarning)
/opt/conda/lib/python3.10/site-packages/transformers/data/metrics/__init__.py:37: FutureWarning: This metric will be removed from the library soon, metrics should be handled with the 🤗 Evaluate library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING, FutureWarning)
/opt/conda/lib/python3.10/site-packages/transformers/data/metrics/__init__.py:31: FutureWarning: This metric will be removed from the library soon, metrics should be han

# Weight Pruning

In [22]:
import torch.nn as nn

def prune_weights(model, pruning_ratio):
    pruned_model=model
    # Function to prune weights of the model
    for name, module in pruned_model.named_modules():
        if isinstance(module, nn.Linear):
            weight = module.weight.data
            # Compute the threshold for pruning
            abs_weight = weight.abs()
            threshold = torch.quantile(abs_weight, pruning_ratio)
            # Zero out weights below the threshold
            mask = abs_weight > threshold
            module.weight.data.mul_(mask)
    return pruned_model

# Prune 20% of the weights
pruned_model= prune_weights(model, pruning_ratio=0.2)

In [23]:
print_size_of_model(pruned_model)

Size (MB): 438.751567


In [24]:
from prettytable import PrettyTable

def count_parameters(model):
    table = PrettyTable(["Modules", "Parameters"])
    total_params = 0
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        params = parameter.numel()
        table.add_row([name, params])
        total_params += params
    print(table)
    print(f"Total Trainable Params: {total_params}")
    return total_params
    
count_parameters(pruned_model)

+---------------------------------------------------------+------------+
|                         Modules                         | Parameters |
+---------------------------------------------------------+------------+
|          bert.embeddings.word_embeddings.weight         |  23440896  |
|        bert.embeddings.position_embeddings.weight       |   393216   |
|       bert.embeddings.token_type_embeddings.weight      |    1536    |
|             bert.embeddings.LayerNorm.weight            |    768     |
|              bert.embeddings.LayerNorm.bias             |    768     |
|     bert.encoder.layer.0.attention.self.query.weight    |   589824   |
|      bert.encoder.layer.0.attention.self.query.bias     |    768     |
|      bert.encoder.layer.0.attention.self.key.weight     |   589824   |
|       bert.encoder.layer.0.attention.self.key.bias      |    768     |
|     bert.encoder.layer.0.attention.self.value.weight    |   589824   |
|      bert.encoder.layer.0.attention.self.value.bi

109483778

In [25]:
time_model_evaluation(pruned_model, configs, tokenizer)

/opt/conda/lib/python3.10/site-packages/transformers/data/processors/glue.py:174: FutureWarning: This processor will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING.format("processor"), FutureWarning)
Evaluating: 100%|██████████| 51/51 [02:41<00:00,  3.17s/it]

{'acc': 0.8504901960784313, 'f1': 0.8960817717206134, 'acc_and_f1': 0.8732859838995224}
Evaluate total time (seconds): 161.7



/opt/conda/lib/python3.10/site-packages/transformers/data/metrics/__init__.py:61: FutureWarning: This metric will be removed from the library soon, metrics should be handled with the 🤗 Evaluate library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING, FutureWarning)
/opt/conda/lib/python3.10/site-packages/transformers/data/metrics/__init__.py:37: FutureWarning: This metric will be removed from the library soon, metrics should be handled with the 🤗 Evaluate library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING, FutureWarning)
/opt/conda/lib/python3.10/site-packages/transformers/data/metrics/__init__.py:31: FutureWarning: This metric will be removed from the library soon, metrics should be han

# Pruning with Dynamic Quantization

In [26]:
import torch.nn as nn

def prune_weights(model, pruning_ratio):
    pruned_model=model
    # Function to prune weights of the model
    for name, module in pruned_model.named_modules():
        if isinstance(module, nn.Linear):
            weight = module.weight.data
            # Compute the threshold for pruning
            abs_weight = weight.abs()
            threshold = torch.quantile(abs_weight, pruning_ratio)
            # Zero out weights below the threshold
            mask = abs_weight > threshold
            module.weight.data.mul_(mask)
    pruned_quantized_model = torch.quantization.quantize_dynamic(
    pruned_model, {torch.nn.Linear}, dtype=torch.qint8
    )
    
    return pruned_quantized_model

# Prune 20% of the weights
pruned_quantized_model= prune_weights(pruned_model, pruning_ratio=0.2)
print(pruned_quantized_model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): QuantizedEmbedding(num_embeddings=30522, embedding_dim=768, dtype=torch.quint8, qscheme=torch.per_channel_affine_float_qparams)
      (position_embeddings): QuantizedEmbedding(num_embeddings=512, embedding_dim=768, dtype=torch.quint8, qscheme=torch.per_channel_affine_float_qparams)
      (token_type_embeddings): QuantizedEmbedding(num_embeddings=2, embedding_dim=768, dtype=torch.quint8, qscheme=torch.per_channel_affine_float_qparams)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(
                in_features=768, out_features=768, bias=True
                (weight_fake_quant): PerChannelMinMaxObserver(
          

In [27]:
print_size_of_model(pruned_quantized_model)

Size (MB): 367.484229


In [28]:
from prettytable import PrettyTable

def count_parameters(model):
    table = PrettyTable(["Modules", "Parameters"])
    total_params = 0
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        params = parameter.numel()
        table.add_row([name, params])
        total_params += params
    print(table)
    print(f"Total Trainable Params: {total_params}")
    return total_params
    
count_parameters(pruned_quantized_model)

+---------------------------------------------------------+------------+
|                         Modules                         | Parameters |
+---------------------------------------------------------+------------+
|             bert.embeddings.LayerNorm.weight            |    768     |
|              bert.embeddings.LayerNorm.bias             |    768     |
|     bert.encoder.layer.0.attention.self.query.weight    |   589824   |
|      bert.encoder.layer.0.attention.self.query.bias     |    768     |
|      bert.encoder.layer.0.attention.self.key.weight     |   589824   |
|       bert.encoder.layer.0.attention.self.key.bias      |    768     |
|     bert.encoder.layer.0.attention.self.value.weight    |   589824   |
|      bert.encoder.layer.0.attention.self.value.bias     |    768     |
|    bert.encoder.layer.0.attention.output.dense.weight   |   589824   |
|     bert.encoder.layer.0.attention.output.dense.bias    |    768     |
|  bert.encoder.layer.0.attention.output.LayerNorm.

85648130

In [29]:
time_model_evaluation(pruned_quantized_model, configs, tokenizer)

/opt/conda/lib/python3.10/site-packages/transformers/data/processors/glue.py:174: FutureWarning: This processor will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING.format("processor"), FutureWarning)
Evaluating: 100%|██████████| 51/51 [02:41<00:00,  3.16s/it]

{'acc': 0.8480392156862745, 'f1': 0.89419795221843, 'acc_and_f1': 0.8711185839523523}
Evaluate total time (seconds): 161.1



/opt/conda/lib/python3.10/site-packages/transformers/data/metrics/__init__.py:61: FutureWarning: This metric will be removed from the library soon, metrics should be handled with the 🤗 Evaluate library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING, FutureWarning)
/opt/conda/lib/python3.10/site-packages/transformers/data/metrics/__init__.py:37: FutureWarning: This metric will be removed from the library soon, metrics should be handled with the 🤗 Evaluate library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING, FutureWarning)
/opt/conda/lib/python3.10/site-packages/transformers/data/metrics/__init__.py:31: FutureWarning: This metric will be removed from the library soon, metrics should be han

In [30]:
"""
Add this code to the END of your Kaggle notebook to export the model.

This will export your pruned + quantized BERT model to ONNX format
for use in your Android app.
"""

import torch
import os

print("=" * 60)
print("  Exporting BERT Model to ONNX for Android")
print("=" * 60)
print()

# ============================================================================
# IMPORTANT: Change this variable name to match your notebook!
# ============================================================================
# Look for the variable that contains your final compressed model.
# It might be called:
#   - pruned_quantized_model
#   - quantized_model
#   - pruned_model
#   - model
# 
# Find the last cell where you applied quantization, and use that variable name.

# YOUR MODEL VARIABLE NAME HERE:
model_to_export = pruned_model  # ← CHANGE THIS IF NEEDED

# ============================================================================
# Export Process
# ============================================================================

print("[1/4] Preparing model for export...")

# Ensure model is in evaluation mode and on CPU
model_to_export = model_to_export.cpu()
model_to_export.eval()

# IMPORTANT: Set return_dict=False to get simple tuple output
# This prevents the "NoneType cannot be traced" error
model_to_export.config.return_dict = False

print("✅ Model ready for export")
print()

# ============================================================================
print("[2/4] Creating dummy inputs...")

# BERT expects 3 inputs: input_ids, attention_mask, token_type_ids
batch_size = 1
seq_length = 128

dummy_input_ids = torch.randint(0, 30522, (batch_size, seq_length), dtype=torch.long)
dummy_attention_mask = torch.ones(batch_size, seq_length, dtype=torch.long)
dummy_token_type_ids = torch.zeros(batch_size, seq_length, dtype=torch.long)

print("✅ Dummy inputs created")
print()

# ============================================================================
print("[3/4] Exporting to ONNX format...")

output_filename = 'bert_pruned_quantized.onnx'

try:
    torch.onnx.export(
        model_to_export,
        (dummy_input_ids, dummy_attention_mask, dummy_token_type_ids),
        output_filename,
        input_names=['input_ids', 'attention_mask', 'token_type_ids'],
        output_names=['logits'],
        dynamic_axes={
            'input_ids': {0: 'batch', 1: 'sequence'},
            'attention_mask': {0: 'batch', 1: 'sequence'},
            'token_type_ids': {0: 'batch', 1: 'sequence'},
            'logits': {0: 'batch'}
        },
        opset_version=14,
        do_constant_folding=True,
        verbose=False
    )
except Exception as e:
    print(f"❌ ONNX export failed with error:")
    print(f"   {str(e)}")
    print()
    print("Trying alternative export method...")
    print()
    
    # Alternative: Script the model using TorchScript first
    print("Attempting TorchScript conversion first...")
    
    # Create a simple wrapper that only returns logits
    import torch.nn as nn
    
    class SimpleLogitsWrapper(nn.Module):
        def __init__(self, bert_model):
            super().__init__()
            self.bert = bert_model.bert  # Get the BERT encoder
            self.classifier = bert_model.classifier  # Get the classification head
            
        def forward(self, input_ids, attention_mask, token_type_ids):
            # Get BERT outputs
            outputs = self.bert(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
            
            # Get pooled output (CLS token representation)
            if isinstance(outputs, tuple):
                pooled_output = outputs[1]  # pooled output is second element
            else:
                pooled_output = outputs.pooler_output
            
            # Apply classifier
            logits = self.classifier(pooled_output)
            return logits
    
    print("Creating simple wrapper model...")
    wrapped_model = SimpleLogitsWrapper(model_to_export)
    wrapped_model.eval()
    
    # Test the wrapper
    print("Testing wrapper model...")
    with torch.no_grad():
        test_output = wrapped_model(dummy_input_ids, dummy_attention_mask, dummy_token_type_ids)
        print(f"   Wrapper output shape: {test_output.shape}")
    
    # Export the wrapped model
    print("Exporting wrapped model to ONNX...")
    torch.onnx.export(
        wrapped_model,
        (dummy_input_ids, dummy_attention_mask, dummy_token_type_ids),
        output_filename,
        input_names=['input_ids', 'attention_mask', 'token_type_ids'],
        output_names=['logits'],
        dynamic_axes={
            'input_ids': {0: 'batch', 1: 'sequence'},
            'attention_mask': {0: 'batch', 1: 'sequence'},
            'token_type_ids': {0: 'batch', 1: 'sequence'},
            'logits': {0: 'batch'}
        },
        opset_version=14,
        do_constant_folding=True,
        verbose=False
    )

print(f"✅ Model exported: {output_filename}")
print()

# ============================================================================
print("[4/4] Checking exported file...")

if os.path.exists(output_filename):
    file_size_mb = os.path.getsize(output_filename) / (1024 * 1024)
    print(f"✅ File exists: {output_filename}")
    print(f"   Size: {file_size_mb:.2f} MB")
    
    # Provide download instructions
    print()
    print("=" * 60)
    print("  ✅ Export Complete!")
    print("=" * 60)
    print()
    print("Next Steps:")
    print("  1. Look in the 'Output' tab on the right →")
    print(f"  2. Find '{output_filename}'")
    print("  3. Click the download icon (⬇️)")
    print("  4. Save the file to your computer")
    print()
    print("Then copy it to:")
    print("  C:\\Users\\User\\Desktop\\Thesis\\Mindforge\\app\\src\\main\\assets\\")
    print()
    print("Finally:")
    print("  • Rebuild your Android app")
    print("  • Run it!")
    print("  • Check logs for: '✅ BERT model loaded successfully!'")
    print()
    
    # Show model details
    print("Model Details:")
    print(f"  • Base: bert-base-uncased (fine-tuned on MRPC)")
    print(f"  • Compression: 20% weighted pruning + 8-bit quantization")
    print(f"  • Format: ONNX (opset 14)")
    print(f"  • Size: {file_size_mb:.2f} MB")
    print(f"  • Expected accuracy: 85-90%")
    print()
    
else:
    print("❌ Export failed - file not found")
    print("   Check the output above for errors")
    print()


  Exporting BERT Model to ONNX for Android

[1/4] Preparing model for export...
✅ Model ready for export

[2/4] Creating dummy inputs...
✅ Dummy inputs created

[3/4] Exporting to ONNX format...
✅ Model exported: bert_pruned_quantized.onnx

[4/4] Checking exported file...
✅ File exists: bert_pruned_quantized.onnx
   Size: 417.94 MB

  ✅ Export Complete!

Next Steps:
  1. Look in the 'Output' tab on the right →
  2. Find 'bert_pruned_quantized.onnx'
  3. Click the download icon (⬇️)
  4. Save the file to your computer

Then copy it to:
  C:\Users\User\Desktop\Thesis\Mindforge\app\src\main\assets\

Finally:
  • Rebuild your Android app
  • Run it!
  • Check logs for: '✅ BERT model loaded successfully!'

Model Details:
  • Base: bert-base-uncased (fine-tuned on MRPC)
  • Compression: 20% weighted pruning + 8-bit quantization
  • Format: ONNX (opset 14)
  • Size: 417.94 MB
  • Expected accuracy: 85-90%

